In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -qqq langchain langchain-community -y
!pip install -qqq langchain langchain-community langchain-core

import warnings

from langchain_community.document_loaders import PyPDFDirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain_community.chains import RetrievalQA
from langchain_community.chains import LLMChain

In [ ]:
loader=PyPDFDirectoryLoader(r"/content/drive/MyDrive/Medical chatbot/data")
docs=loader.load()

In [ ]:
print(f"Number of documents: {len(docs)}")

Number of documents: 95


##creating chunks

In [ ]:
text_spliteter=RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks=text_spliteter.split_documents(docs)

In [ ]:

embeddings=SentenceTransformerEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")
vectorstore=Chroma.from_documents(chunks, embeddings)

/tmp/ipykernel_7670/2669616063.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=SentenceTransformerEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
query="What are the symptoms of diabetes?"
serch_results=vectorstore.similarity_search(query)
print(serch_results)

[Document(metadata={'producer': 'Acrobat Distiller 6.0.1 for Macintosh', 'keywords': 'heart disease, prevention, risk factors, chd, coronary artery disease, corornary heart disease, cad', 'page_label': '38', 'page': 37, 'author': 'NHLBI', 'creator': 'QuarkXPress(tm) 6.5', 'source': '/content/drive/MyDrive/Medical chatbot/data/healthyheart.pdf', 'subject': 'Heart disease', 'moddate': '2006-02-23T09:58:15-05:00', 'total_pages': 95, 'creationdate': '2006-02-16T11:30:29-05:00', 'title': 'Your Guide to A Healthy Heart'}, page_content='more than 9 pounds are also more likely to develop type 2 diabetes\nlater in life.\nSymptoms of diabetes may include fatigue, nausea, frequent urination,\nunusual thirst, weight loss, blurred vision, frequent infections, and\nslow healing of sores. But type 2 diabetes develops gradually and\nsometimes has no symptoms. Even if you have no symptoms of'), Document(metadata={'creationdate': '2006-02-16T11:30:29-05:00', 'author': 'NHLBI', 'creator': 'QuarkXPress(tm

In [ ]:
retriver=vectorstore.as_retriever(search_kwargs={"k":5})
retriver.invoke(query)

[Document(metadata={'page_label': '38', 'title': 'Your Guide to A Healthy Heart', 'producer': 'Acrobat Distiller 6.0.1 for Macintosh', 'source': '/content/drive/MyDrive/Medical chatbot/data/healthyheart.pdf', 'subject': 'Heart disease', 'creationdate': '2006-02-16T11:30:29-05:00', 'page': 37, 'creator': 'QuarkXPress(tm) 6.5', 'author': 'NHLBI', 'moddate': '2006-02-23T09:58:15-05:00', 'total_pages': 95, 'keywords': 'heart disease, prevention, risk factors, chd, coronary artery disease, corornary heart disease, cad'}, page_content='more than 9 pounds are also more likely to develop type 2 diabetes\nlater in life.\nSymptoms of diabetes may include fatigue, nausea, frequent urination,\nunusual thirst, weight loss, blurred vision, frequent infections, and\nslow healing of sores. But type 2 diabetes develops gradually and\nsometimes has no symptoms. Even if you have no symptoms of'),
 Document(metadata={'total_pages': 95, 'author': 'NHLBI', 'keywords': 'heart disease, prevention, risk factor

In [ ]:
llm=LlamaCpp(model_path="/content/drive/MyDrive/Medical chatbot/model/ggml-model-Q4_K_M.gguf",temperature=0.2,max_tokens=2000,top_p=1)

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /content/drive/MyDrive/Medical chatbot/model/ggml-model-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = models
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:               

In [ ]:
template="""
<|context|>
You are an medical assistant that follows the instructions and generate the accurate response based on the query and the context provided.
Be truthful and give correct answers.
</s>
<|user|>
</s>
{query}
</s>
<|assistant|>
"""


In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

In [ ]:
prompt=PromptTemplate.from_template(template)

In [ ]:
RAG_Chain=(
    {"context":retriver,"query":RunnablePassthrough()}
    |prompt
    |llm
    |StrOutputParser()
)

In [ ]:
query= "who is at risk of diabates"

In [ ]:
response=RAG_Chain.invoke(query)

llama_perf_context_print:        load time =   30864.16 ms
llama_perf_context_print: prompt eval time =   30859.21 ms /    73 tokens (  422.73 ms per token,     2.37 tokens per second)
llama_perf_context_print:        eval time =   21131.72 ms /    43 runs   (  491.44 ms per token,     2.03 tokens per second)
llama_perf_context_print:       total time =   52026.64 ms /   116 tokens
llama_perf_context_print:    graphs reused =         48


In [ ]:
print(response)

The risk of developing diabetes increases with age. Other factors that can increase the risk of developing diabetes include family history of diabetes, overweight or obesity, physical inactivity, and history of gestational diabetes.


In [ ]:
import sys

while True:
  user_input=input(f"Input_query:")
  if user_input=='exit':
    print('Exited..')
    sys.exit()
  if user_input=="":
    continue
  result=RAG_Chain.invoke(user_input)
  print("Answer =",result)

Input_query:who is at risk of diabates


Llama.generate: 72 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =   30864.16 ms
llama_perf_context_print: prompt eval time =    7893.37 ms /     2 tokens ( 3946.68 ms per token,     0.25 tokens per second)
llama_perf_context_print:        eval time =   61342.64 ms /   126 runs   (  486.85 ms per token,     2.05 tokens per second)
llama_perf_context_print:       total time =   61946.49 ms /   128 tokens
llama_perf_context_print:    graphs reused =        122


Answer = The risk of diabetes is determined by a number of factors, including family history, age, lifestyle, and body mass index. Diabetes risk is higher for people who have a parent or sibling with the disease. The risk also increases with age, especially after age 45. Lifestyle factors that can increase the risk of diabetes include being overweight or obese, not getting enough physical activity, and smoking. Having a body mass index (BMI) of 25 kg/m2 or higher puts you at risk for diabetes. The more your BMI is above this, the greater your risk.
Input_query:exit
Exited..


SystemExit: 

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
